In [ ]:
import pickle
from pathlib import Path

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
DATA_DIR = Path(".")
with open(DATA_DIR / "train_data.pickle", "rb") as f:
    train = pickle.load(f)
X = train["X"]
y = train["y"]

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("y head:", y[:10])
print("unique classes:", np.unique(y))

In [ ]:
print("NaN count in X:", np.isnan(X).sum())
print("NaN count in y:", np.isnan(y.astype(float)).sum())

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ["accuracy", "f1_weighted", "balanced_accuracy"]

pipe_lr = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(max_iter=8000, random_state=42, solver="lbfgs"),
        ),
    ]
)
rf = RandomForestClassifier(n_estimators=300, random_state=42)

candidates = {"pipeline_lr": pipe_lr, "random_forest": rf}
cv_results = {}
for name, model in candidates.items():
    cv_results[name] = {}
    print(name)
    for m in metrics:
        scores = cross_val_score(model, X, y, cv=cv, scoring=m)
        cv_results[name][m] = scores
        print(f"  {m}: {scores.mean():.4f} (+/- {2 * scores.std():.4f})")

In [ ]:
primary = "f1_weighted"
best_name = max(candidates, key=lambda n: cv_results[n][primary].mean())
print("best by", primary, ":", best_name)
best_model = candidates[best_name]

In [ ]:
best_model.fit(X, y)
out_path = DATA_DIR / "model_best.pickle"
with open(out_path, "wb") as f:
    pickle.dump(best_model, f, protocol=4)
print("saved:", out_path.resolve())

Docker команды

```bash
docker build -t lab2-dev .
docker run --rm -v "$(pwd)":/lab2 lab2-dev \
  -d /lab2 -m model_best.pickle -i test_data.pickle -a all
```